In [ ]:
!git clone -q https://github.com/Mikayla-ds2/surge-sense.git

In [ ]:
%cd surge-sense

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path
%pip install pmdarima
import pmdarima as pm
%pip install statsmodels
from statsmodels.tsa.seasonal import seasonal_decompose, STL

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/daily_visits.csv')

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data['admission_day'] = pd.to_datetime(data['admission_day'])

In [ ]:
plot = data.plot(x='admission_day', y='visit_count', legend=False, figsize=(20, 8))
plt.xticks(rotation=90, fontsize=10)
plt.title('# of Visits')
plt.xlabel('Time')
plt.ylabel('# of Visits')
plt.show()

In [ ]:
data['month_day'] = data['admission_day'].dt.strftime('%m-%d')
data.head()

In [ ]:
data['month'] = data['admission_day'].dt.strftime('%m')
data['day'] = data['admission_day'].dt.strftime('%d')
data['weekday'] = data['admission_day'].dt.day_name()
data.head()

In [ ]:
sum_visits = data.groupby('month_day', as_index=False)['visit_count'].sum()
agg_visits_sum = sum_visits.sort_values('month_day')

display(agg_visits_sum.loc[agg_visits_sum['visit_count'].nlargest(10).index])

plot = agg_visits_sum.plot(x='month_day', y='visit_count', legend=False, figsize=(20, 8))
plt.xticks(rotation=90, fontsize=10)
plt.title('Total Visits by Month and Day')
plt.xlabel('Month-Day')
plt.ylabel('Total Visit Count')
plt.show()

In [ ]:
avg_visits = data.groupby('month_day', as_index=False)['visit_count'].mean()
agg_visits_avg = avg_visits.sort_values('month_day')

display(agg_visits_avg.loc[agg_visits_avg['visit_count'].nlargest(10).index])

plot = agg_visits_avg.plot(x='month_day', y='visit_count', legend=False, figsize=(20, 8))
plt.xticks(rotation=90, fontsize=10)
plt.title('Average Visits by Month and Day')
plt.xlabel('Month-Day')
plt.ylabel('Average Visit Count')
plt.show()

In [ ]:
monthly_avg_visits = data.groupby('month', as_index=False)['visit_count'].mean()
agg_monthly_avg_visits = monthly_avg_visits.sort_values('month')

display(agg_monthly_avg_visits.loc[agg_monthly_avg_visits['visit_count'].nsmallest(12).index])

plot = agg_monthly_avg_visits.plot(x='month', y='visit_count', legend=False, figsize=(16, 6))
plt.xticks(rotation=90, fontsize=10)
plt.title('Average Visits by Month')
plt.xlabel('Month')
plt.ylabel('Average Visit Count')
plt.show()

In [ ]:
data['quarter'] = pd.PeriodIndex(data.admission_day, freq='Q')
data.head()

In [ ]:
quarterly_avg_visits = data.groupby('quarter', as_index=False)['visit_count'].mean()
agg_visits = quarterly_avg_visits.sort_values('quarter')

display(agg_visits.loc[agg_visits['visit_count'].nlargest(10).index])    

plot = agg_visits.plot(x='quarter', y='visit_count', legend=False, figsize=(12, 6))
plt.xticks(rotation=90, fontsize=10)
plt.title('Average Visits by Quarter')
plt.xlabel('Quarter')
plt.ylabel('Average Visit Count')
plt.show()

In [ ]:
iso = data['admission_day'].dt.isocalendar()
data['week_sequential'] = iso['year'].astype(str) + '-W' + iso['week'].astype(str).str.zfill(2)
data['week_of_year'] = iso['week'] # week 1-52 of the year; will repeat
data.head()

In [ ]:
weekly_avg_visits = data.groupby('week_sequential', as_index=False)['visit_count'].mean()
agg_visits = weekly_avg_visits.sort_values('week_sequential')

display(agg_visits.loc[agg_visits['visit_count'].nlargest(10).index])    

plot = agg_visits.plot(x='week_sequential', y='visit_count', legend=False, figsize=(12, 6))
plt.xticks(rotation=90, fontsize=10)
plt.title('Average Visits by Week')
plt.xlabel('Week')
plt.ylabel('Average Visit Count')
plt.show()

In [ ]:
weekly_avg_visits = data.groupby('week_of_year', as_index=False)['visit_count'].mean()
agg_visits = weekly_avg_visits.sort_values('week_of_year')

display(agg_visits.loc[agg_visits['visit_count'].nlargest(10).index])    

plot = agg_visits.plot(x='week_of_year', y='visit_count', legend=False, figsize=(12, 6))
plt.xticks(rotation=90, fontsize=10)
plt.title('Average Visits by Week')
plt.xlabel('Week')
plt.ylabel('Average Visit Count')
plt.show()

In [ ]:
data.info()

In [ ]:
# quick bar plot for average visit count by weekday
mean_visits = data.groupby('weekday')['visit_count'].mean()
plt.figure(figsize=(12, 6))
mean_visits.plot(kind='bar')
plt.title('Average visit count by weekday')
plt.xlabel('Weekday')
plt.ylabel('Average Visit Count')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
data.visit_count.describe()

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(data['visit_count'], bins=15)
plt.show()

first outlier detection using a threshold: z=2 are varied activity days; not necessarily entire outliers. however, August 10th, flagging for both years, needs to be investigated – which I will. 

In [ ]:
# outlier detection - 
df = data.copy()
df['z_score'] = np.abs((df['visit_count'] - df['visit_count'].mean()) / df['visit_count'].std())
outliers_z2_score = df[df['z_score'] > 2]
z_outlier_percent = (len(outliers_z2_score) / len(df)) * 100

print(f"Detected {len(outliers_z2_score)} outliers ({z_outlier_percent:.2f}% of the dataset) using z-scores; threshold = 2.\n")

ax = data.plot(x='admission_day', y='visit_count', label="Time Series", figsize=(20, 6))
outlier_dates = matplotlib.dates.date2num(outliers_z2_score['admission_day'])
ax.scatter(outlier_dates, outliers_z2_score['visit_count'], color='red', label='Detected outliers')
ax.legend()
plt.show()

second outlier detection using threshold: z=3 actual worth reporting outliers – will likely NaN these due to Prophet's needs & keeping confidence band honest.

In [ ]:
# outlier detection
df = data.copy()
df['z_score'] = np.abs((df['visit_count'] - df['visit_count'].mean()) / df['visit_count'].std())
outliers_z3_score = df[df['z_score'] > 3]
z_outlier_percent = (len(outliers_z3_score) / len(df)) * 100

print(f"Detected {len(outliers_z3_score)} outliers ({z_outlier_percent:.2f}% of the dataset) using z-scores; threshold = 3.\n")

ax = data.plot(x='admission_day', y='visit_count', label="Time Series", figsize=(20, 6))
outlier_dates = matplotlib.dates.date2num(outliers_z3_score['admission_day'])
z2_outlier_dates = matplotlib.dates.date2num(outliers_z2_score['admission_day'])
ax.scatter(outlier_dates, outliers_z3_score['visit_count'], color='red', label='Detected outliers')
ax.legend()
plt.show()

In [ ]:
z3_outlier_days = outliers_z3_score[['admission_day']].copy()
z3_outlier_days['admission_day'] = z3_outlier_days['admission_day'].dt.strftime('%Y-%m-%d')

z2_outlier_days = outliers_z2_score[['admission_day']].copy()
z2_outlier_days['admission_day'] = z2_outlier_days['admission_day'].dt.strftime('%Y-%m-%d')
display(z3_outlier_days)
display(z2_outlier_days)

In [ ]:
data.head()

In [ ]:
q1 = df['visit_count'].quantile(0.25)
q3 = df['visit_count'].quantile(0.75)
iqr = q3 - q1 # 5
lower_bound = q1 - 1.5 * iqr # 5.5
upper_bound = q3 + 1.5 * iqr # 25.5
outliers_iqr = df[(df['visit_count'] < lower_bound) | (df['visit_count'] > upper_bound)]
iqr_percent = (len(outliers_iqr) / df.shape[0]) * 100

print(f"Detected {len(outliers_iqr)} outliers ({iqr_percent:.2f}% of the dataset) in visit-count column using IQR.\n")
fig, ax = plt.subplots(figsize=(20, 6))
ax.plot(df['admission_day'], df['visit_count'], label='Time Series')
ax.scatter(outliers_iqr['admission_day'], outliers_iqr['visit_count'],
           color='red', label='Detected outliers')
ax.legend()
plt.show()

In [ ]:
# building the side-by-side boxplot & series plots for each quarter
base_dir = Path('quarterly_double_plots')
base_dir.mkdir(exist_ok=True)

quarters = data['quarter'].dropna().sort_values().unique()
quarterly_plots = {}

def plot_quarter(quarter):
    quarter_data = data.loc[data['quarter'].eq(quarter)].sort_values('admission_day')
    quarter_label = str(quarter)

    fig, (ax_box, ax_line) = plt.subplots(
        1,
        2,
        sharey=True,
        gridspec_kw={'width_ratios': [1, 5]},
        figsize=(10, 4)
    )

    ax_box.boxplot(
        quarter_data['visit_count'],
        tick_labels=[quarter_label],
        widths=0.5,
        patch_artist=True,
        showfliers=True
    )
    ax_box.set_title(quarter_label)
    ax_box.set_ylabel('Visit Count')
    ax_box.tick_params(labelsize=10)

    ax_line.plot(
        quarter_data['admission_day'],
        quarter_data['visit_count'],
        marker='o',
        markersize=3,
        linewidth=1.5
    )
    ax_line.set_xlabel('Date')
    ax_line.set_title(f'{quarter_label} Daily Trace')
    ax_line.tick_params(labelleft=False)

    fig.suptitle(f'{quarter_label}: Distribution vs. Daily Trace', fontweight='bold')
    fig.subplots_adjust(wspace=0.05)
    fig.autofmt_xdate(rotation=45)
    fig.tight_layout()
    fig.savefig(base_dir / f'{quarter_label}.png', bbox_inches='tight')
    return fig, ax_box, ax_line


for quarter in quarters:
    quarterly_plots[str(quarter)] = plot_quarter(quarter)

plt.show()


In [ ]:
# shared outlier plot (time series with indicated outliers from all methodologies)
fig, ax = plt.subplots(figsize=(20, 6))
ax.plot(df['admission_day'], df['visit_count'], label='Time Series')
ax.scatter(outliers_iqr['admission_day'], outliers_iqr['visit_count'],
           color='red', label='Detected outliers')
ax.scatter(outlier_dates, outliers_z3_score['visit_count'], color='red')
ax.scatter(z2_outlier_dates, outliers_z2_score['visit_count'], color='red')
ax.legend()
plt.show()

In [ ]:
# NaN-ing out the 8 outliers that have been confirmed at least twice
outliers_iqr.head(10)

In [ ]:
# seasonal decomposition - months
decomp = seasonal_decompose(data.visit_count, model='additive', period=12)
decomp.plot()
plt.show()

In [ ]:
# seasonal decomposition - quarters
decomp_q = seasonal_decompose(data.visit_count, model='additive', period=4)
decomp_q.plot()
plt.show()

In [ ]:
# seasonal_decomposition - weeks
decomp_w = seasonal_decompose(data.visit_count, model='additive', period=52)
decomp_w.plot()
plt.show()